# MCP From Scratch -- interactive notebook

This notebook talks to the from-scratch MCP server (`mcp_server.py`) while it runs as an
**independent background process** instead of a subprocess this notebook spawns itself. That's
the difference from `demo.py`: here the server survives across cells, kernel restarts, and even
reconnects from a second notebook, because nothing about it is tied to this process's lifetime.

**Before running any cell below**, start the server from a terminal in this directory:

```bash
make server-bg
```

That prints the two named-pipe paths the server is listening on (`.pipes/request.pipe` and
`.pipes/response.pipe`) -- see `mcp_server.py`'s `--pipe-in`/`--pipe-out` mode and
`MCPClient.connect()` in `mcp_client.py` for how this works instead of the usual
spawn-a-subprocess-over-stdio path. Check `make server-status` any time, and `make server-stop`
(or the last cell here) when you're done.

In [ ]:
from mcp_client import MCPClient, MCPError

REQUEST_PIPE = ".pipes/request.pipe"
RESPONSE_PIPE = ".pipes/response.pipe"

# verbose=False here because the server's own wire trace is already going to server.log
# (see the last cell for how to peek at it) -- no need to duplicate it into the notebook.
client = MCPClient.connect(REQUEST_PIPE, RESPONSE_PIPE, verbose=False)
info = client.initialize()
info

## Discover tools

Same `tools/list` call `demo.py` makes -- each tool's `inputSchema` is plain JSON Schema, written
by hand in `mcp_server.py`'s `TOOLS` list (the real SDK generates this from type hints instead).

In [ ]:
client.list_tools()

## Call tools

Because the server persists in the background, tasks added here stick around in `tasks.json`
even if you restart this notebook's kernel -- re-run the connect cell above and they're still
there, which wouldn't be true of `demo.py`'s subprocess-per-run model.

In [ ]:
print(client.call_tool("mcp_add_task", {"title": "learn MCP from a notebook"}))

In [ ]:
print(client.call_tool("mcp_list_tasks"))

In [ ]:
print(client.call_tool("mcp_complete_task", {"task_id": 1}))
print(client.call_tool("mcp_list_tasks"))

## Protocol-level errors

An unknown tool name or a missing required argument never reaches `_call_tool` at all --
`dispatch()` in `mcp_server.py` rejects both as JSON-RPC errors (code `-32602`,
`INVALID_PARAMS`) before any tool logic runs. `MCPClient` raises these as `MCPError`. Compare this
with a tool that *runs* but reports failure (e.g. completing a task id that doesn't exist) --
that comes back as an ordinary successful call whose text says it failed, not an exception; see
the README's "Error handling: two kinds, easy to conflate" section.

In [ ]:
try:
    client.call_tool("mcp_delete_everything", {})
except MCPError as exc:
    print(f"caught MCPError as expected: {exc}")

In [ ]:
try:
    client.call_tool("mcp_add_task", {})
except MCPError as exc:
    print(f"caught MCPError as expected: {exc}")

## Resources

The other primitive this project implements: a `*/list` + `*/read` shape (rather than tools'
`*/list` + `*/call`) for read-only content a client can pull in.

In [ ]:
client.list_resources()

In [ ]:
print(client.read_resource("tasks://all"))

## Peek at the wire trace

Every request/response this notebook has sent so far was also logged, raw, on the server side --
`server.log` (stdout+stderr of the background process, redirected there by `make server-bg`).

In [ ]:
!tail -20 server.log

## Shutting down

`client.close()` closes this client's end of the request pipe, the server sees EOF (the same
shutdown convention as the normal stdio transport) and exits -- so this also ends the background
server, not just this notebook's connection to it. Start a fresh one with `make server-bg` next
time. If you'd rather leave the server running and just stop using it from this notebook, skip
this cell and run `make server-stop` from a terminal whenever you're ready instead.

In [ ]:
client.close()